In [2]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


In [3]:
spark.sql("show databases").show()

+---------+
|namespace|
+---------+
|  default|
| spark_db|
+---------+



In [4]:
spark.sql("show tables in spark_db").show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
| spark_db|    offline_students|      false|
| spark_db|offline_students_raw|      false|
+---------+--------------------+-----------+



In [8]:
"""
Load students_online.csv file and load into online_students table
"""
from pyspark.sql.types import StructField, StructType, IntegerType, StringType, ArrayType, MapType # type: ignore

# {"ID":"101","FirstName":"Prashant","LastName":"Pandey",
# "Address":{"AddressLine1":"D104 Gopalan Squire","AddressLine2":"Whitefield","City":"Bangalore","State":"Karnataka","Country":"India","Pin":"560001"},
# "Skills":[{"Skill":"Apache Spark","YearsOfExperience":"5"},{"Skill":"Apache Kafka","YearsOfExperience":"6"}],
# "Contacts":{"phone":"9823128923","email":"xyz@abc.com"}}

online_students_schema = StructType([
    StructField("ID", StringType()),
    StructField("FirstName", StringType()),
    StructField("LastName", StringType()),
    StructField("Address", StructType([
        StructField("AddressLine1", StringType()),
        StructField("AddressLine2", StringType()),
        StructField("City", StringType()),
        StructField("State", StringType()),
        StructField("Country", StringType()),
        StructField("Pin", StringType())
    ])),
    StructField("Skills", ArrayType(StructType([
        StructField("Skill", StringType()),
        StructField("YearsOfExperience", StringType())
    ]))),
    StructField("Contacts", MapType(StringType(), StringType()))
])

online_students_df = spark.read.format("json")\
                            .schema(online_students_schema)\
                            .load(path = "/home/jovyan/work/data/students_online.json")

online_students_df.show()
online_students_df.printSchema()

+---+---------+--------+--------------------+--------------------+--------------------+
| ID|FirstName|LastName|             Address|              Skills|            Contacts|
+---+---------+--------+--------------------+--------------------+--------------------+
|101| Prashant|  Pandey|{D104 Gopalan Squ...|[{Apache Spark, 5...|{phone -> 9823128...|
|102|    David|  Turner|{109 Park Street,...|[{Java, 12}, {Spr...|{phone -> 9873145...|
|103|    Katie|Mcloskey|{9th Avenue, Dors...|[{SQL, 12}, {PL/S...|{email -> ert89@a...|
|104|   Nasima|  Khatun|{G105 MG Tower, B...|[{Hadoop, 3}, {Ap...|{email -> magt23@...|
|105|   Pritam|    Jain|{M206 Richmond To...|[{Python, 10}, {S...|{whatsapp -> 6924...|
+---+---------+--------+--------------------+--------------------+--------------------+

root
 |-- ID: string (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Address: struct (nullable = true)
 |    |-- AddressLine1: string (nullable = true)

In [9]:
online_students_df.write.mode("overwrite").saveAsTable("spark_db.online_students")

In [10]:
spark.sql("SELECT * FROM spark_db.online_students").show()

+---+---------+--------+--------------------+--------------------+--------------------+
| ID|FirstName|LastName|             Address|              Skills|            Contacts|
+---+---------+--------+--------------------+--------------------+--------------------+
|101| Prashant|  Pandey|{D104 Gopalan Squ...|[{Apache Spark, 5...|{phone -> 9823128...|
|102|    David|  Turner|{109 Park Street,...|[{Java, 12}, {Spr...|{phone -> 9873145...|
|103|    Katie|Mcloskey|{9th Avenue, Dors...|[{SQL, 12}, {PL/S...|{email -> ert89@a...|
|104|   Nasima|  Khatun|{G105 MG Tower, B...|[{Hadoop, 3}, {Ap...|{email -> magt23@...|
|105|   Pritam|    Jain|{M206 Richmond To...|[{Python, 10}, {S...|{whatsapp -> 6924...|
+---+---------+--------+--------------------+--------------------+--------------------+



In [ ]:
"""
2. Requirement
Perform the following analysis

    What is country wise student count.
    Find all students with more than 1 years of Spark knowledge
    Find all students who didn't provide phone or whatsapp
"""

# What is country wise student count.
spark.sql("""
    SELECT address.country, COUNT(*) AS student_count
    FROM spark_db.online_students
    GROUP BY address.country
""").show()

+----------------+-------------+
|         country|student_count|
+----------------+-------------+
|           India|            3|
|        Engaland|            1|
|Northern Ireland|            1|
+----------------+-------------+



In [16]:
# Find all students with more than 1 years of Spark knowledge

spark.sql("""
    WITH online_students_skills AS(
        SELECT id, firstname, lastname, EXPLODE(skills) AS skills
        FROM spark_db.online_students
    )
    SELECT id, firstname, lastname, skills.skill, skills.yearsofexperience
    FROM online_students_skills
    WHERE UPPER(skills.skill) LIKE '%SPARK%' AND skills.yearsofexperience > 1
""").show()

+---+---------+--------+------------+-----------------+
| id|firstname|lastname|       skill|yearsofexperience|
+---+---------+--------+------------+-----------------+
|101| Prashant|  Pandey|Apache Spark|                5|
|104|   Nasima|  Khatun|Apache Spark|                2|
|105|   Pritam|    Jain|Apache Spark|                3|
+---+---------+--------+------------+-----------------+



In [17]:
# Find all students who didn't provide phone or whatsapp

spark.sql("""
    SELECT id, firstname, lastname, contacts['email']
    FROM spark_db.online_students
    WHERE contacts['phone'] IS null AND contacts['whatsapp'] IS null
""").show()

+---+---------+--------+---------------+
| id|firstname|lastname|contacts[email]|
+---+---------+--------+---------------+
|103|    Katie|Mcloskey|  ert89@abc.com|
|104|   Nasima|  Khatun| magt23@abc.com|
+---+---------+--------+---------------+

